# **Actividad 3. Aprendizaje supervisado con PySpark**

### **Curso: TC5057 - Análisis de Grandes Volúmenes de Datos**
#### **Tecnológico de Monterrey - MNA**
##### **Profesor: Dr. Iván Olmos Pineda**

---

| Nombre | Matrícula |
|--------|----------|
| Ana Bonavides Aguilar | A01423281 |

**Modalidad:** Individual

**Dataset:** GTEx Analysis V10 (Genotype-Tissue Expression) - perfil de expresión génica humana

## Contexto del problema

En el proyecto de equipo trabajo con el dataset GTEx, que mide expresión génica (TPM) en muestras de tejido humano sano. La muestra M del equipo se construyó particionando por **grupo de tejido** (5 grupos: Nervioso, Hematopoyético, Cardiovascular, Musculoesquelético, Visceral/Metabólico) y **sexo biológico**.

Para esta actividad voy a usar como variable objetivo el **grupo de tejido** (`TISSUE_GROUP`, 5 clases). La pregunta de aprendizaje supervisado es:

> ¿Puede un modelo identificar el grupo de tejido al que pertenece una muestra a partir de su perfil de expresión génica?

Esto tiene sentido biológico: cada tejido expresa subconjuntos distintos de genes, así que la firma de expresión debería ser discriminativa. Si esto funciona en GTEx (línea base sana), después se podría usar el mismo enfoque para detectar desvíos en muestras anómalas (por ejemplo, astronautas).

## 1. Introducción teórica

### 1.1 Aprendizaje supervisado

El **aprendizaje supervisado** es la rama del aprendizaje automático en la que el modelo aprende a partir de pares (entrada, salida) etiquetados. Dada una matriz de features $X \in \mathbb{R}^{n \times d}$ y un vector de etiquetas $y$, el objetivo es estimar una función $f$ tal que $f(X) \approx y$ y que generalice bien a datos nuevos.

Dos tipos principales:

- **Clasificación:** la etiqueta $y$ es discreta (categorías). Ejemplo: identificar a qué tejido pertenece una muestra.
- **Regresión:** la etiqueta $y$ es continua. Ejemplo: predecir el nivel de expresión de un gen.

El pipeline básico:
1. Preparar datos (limpiar, transformar, seleccionar features).
2. Dividir en conjunto de entrenamiento y prueba.
3. Entrenar el modelo en el conjunto de entrenamiento.
4. Evaluar en el conjunto de prueba con métricas apropiadas.
5. Interpretar resultados y, si es necesario, iterar.

### 1.2 Algoritmos representativos en la literatura

| Familia | Algoritmo | Idea principal |
|---------|-----------|----------------|
| Lineales | Regresión logística | Modela $P(y\|X)$ como una función logística de una combinación lineal de features. |
| Lineales | SVM (Support Vector Machines) | Encuentra el hiperplano que maximiza el margen entre clases. |
| Árboles | Decision Tree | Divide recursivamente el espacio de features en regiones puras por clase. |
| Ensembles | Random Forest | Bagging de muchos árboles entrenados sobre subconjuntos aleatorios. |
| Ensembles | Gradient Boosting (GBT) | Entrena árboles secuencialmente, cada uno corrigiendo los errores del anterior. |
| Redes neuronales | Multilayer Perceptron (MLP) | Composición de capas no lineales. Capta relaciones complejas a costa de interpretabilidad. |
| Bayesianos | Naive Bayes | Asume independencia condicional entre features dado la clase. |
| Basados en instancias | kNN | Clasifica según los k vecinos más cercanos en el espacio de features. |

### 1.3 Algoritmos disponibles en PySpark

PySpark expone implementaciones distribuidas en `pyspark.ml`. Las relevantes para clasificación supervisada son:

**`pyspark.ml.classification`:**
- `LogisticRegression` (binaria y multinomial)
- `DecisionTreeClassifier`
- `RandomForestClassifier`
- `GBTClassifier` (solo binaria en versiones < 3.5)
- `MultilayerPerceptronClassifier`
- `LinearSVC` (binaria)
- `NaiveBayes`
- `OneVsRest` (envoltorio para extender clasificadores binarios a multi-clase)

**`pyspark.ml.regression`:**
- `LinearRegression`, `GeneralizedLinearRegression`
- `DecisionTreeRegressor`, `RandomForestRegressor`, `GBTRegressor`
- `IsotonicRegression`

Todas se integran con `Pipeline` y con transformadores como `VectorAssembler`, `StringIndexer` y `StandardScaler`.

### 1.4 Algoritmos elegidos para esta actividad

Voy a entrenar y comparar **dos** modelos:

1. **Random Forest Classifier** - modelo no lineal basado en ensembles de árboles. Es robusto al ruido, no requiere escalado de features y entrega importancias de variables. Es buen punto de partida cuando hay muchas features (genes) y pocas muestras.
2. **Logistic Regression (multinomial)** - modelo lineal probabilístico. Sirve como baseline y para contrastar el desempeño contra un modelo no lineal. Requiere escalado, así que aprovecho para introducir `StandardScaler` en el pipeline.

La comparación me va a ayudar a contestar: ¿gana la capacidad no lineal de RF, o las relaciones gen-tejido son lo suficientemente lineales en escala TPM como para que LR sea una buena opción?

## 2. Selección de datos

En esta sección voy a dividir el dataset GTEx completo, construir la muestra M (siguiendo la misma lógica del proyecto de equipo) y luego una muestra M' más pequeña por muestreo estratificado.

### 2.1 Setup: imports y sesión de Spark

In [47]:
import sys, os

# apunto el JVM de Spark al mismo Python que ejecuta el notebook
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType
from pyspark.sql.functions import split as spark_split

# imports de ML
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

import pandas as pd
import numpy as np
import random

# importo las constantes globales del proyecto (rutas + seed + sample step)
sys.path.insert(0, os.path.abspath('../src'))
from GlobalVariables import (
    FILE_PATH, SAMPLE_ATTRS_PATH, SUBJECT_PHENO_PATH, RANDOM_SEED
)

# fijo seeds para reproducibilidad
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f'TPM file:           {FILE_PATH}')
print(f'Sample attributes:  {SAMPLE_ATTRS_PATH}')
print(f'Subject phenotypes: {SUBJECT_PHENO_PATH}')
print(f'Seed:               {RANDOM_SEED}')

TPM file:           /Users/annie/mna/big-data/src/data/GTEx_Analysis_2022-06-06_v10_RNASeQCv2.4.2_gene_tpm_non_lcm.gct
Sample attributes:  /Users/annie/mna/big-data/src/data/GTEx_Analysis_v10_Annotations_SampleAttributesDS.txt
Subject phenotypes: /Users/annie/mna/big-data/src/data/GTEx_Analysis_v10_Annotations_SubjectPhenotypesDS.txt
Seed:               42


In [ ]:
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('GTEx_Actividad3_A01423281') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

spark.conf.set('spark.sql.repl.eagerEval.enabled', True)
spark

### 2.2 Carga de metadatos

Para etiquetar cada muestra con su grupo de tejido necesito los archivos de anotaciones del proyecto GTEx. Estoy reutilizando la lógica del notebook de particionamiento del equipo.

In [49]:
# cargo SampleAttributes y me quedo solo con muestras RNASEQ (las que están en el archivo TPM)
sa_df = spark.read.csv(SAMPLE_ATTRS_PATH, sep='\t', header=True) \
    .select('SAMPID', 'SMTS', 'SMAFRZE') \
    .filter(F.col('SMAFRZE') == 'RNASEQ')

# extraigo el ID del donante a partir del SAMPID (ej: GTEX-1117F-0005-... -> GTEX-1117F)
sa_df = sa_df.withColumn(
    'SUBJID',
    F.regexp_extract(F.col('SAMPID'), r'^(GTEX-[^-]+)', 1)
)

print(f'Muestras RNASEQ: {sa_df.count():,}')
sa_df.show(3, truncate=False)

Muestras RNASEQ: 19,788
+-----------------------------+-----+-------+----------+
|SAMPID                       |SMTS |SMAFRZE|SUBJID    |
+-----------------------------+-----+-------+----------+
|GTEX-1117F-0005-SM-HL9SH     |Blood|RNASEQ |GTEX-1117F|
|GTEX-1117F-0011-R10b-SM-GI4VE|Brain|RNASEQ |GTEX-1117F|
|GTEX-1117F-0011-R11b-SM-GIN8R|Brain|RNASEQ |GTEX-1117F|
+-----------------------------+-----+-------+----------+
only showing top 3 rows


In [50]:
# cargo fenotipos de donantes para obtener el sexo
sp_df = spark.read.csv(SUBJECT_PHENO_PATH, sep='\t', header=True) \
    .select('SUBJID', 'SEX')

print(f'Donantes: {sp_df.count():,}')
sp_df.show(3)

Donantes: 981
+----------+---+
|    SUBJID|SEX|
+----------+---+
|GTEX-1117F|  2|
|GTEX-111CU|  1|
|GTEX-111FC|  1|
+----------+---+
only showing top 3 rows


In [51]:
# uno ambos para tener tejido + sexo por muestra
meta_df = sa_df.join(sp_df, on='SUBJID', how='inner')

# mapeo SMTS a los 5 grupos de tejido del proyecto de equipo
tissue_group_col = F.when(F.col('SMTS').isin('Brain', 'Nerve'), 'Nervioso') \
    .when(F.col('SMTS').isin('Blood', 'Bone Marrow', 'Spleen'), 'Hematopoyetico') \
    .when(F.col('SMTS').isin('Heart', 'Blood Vessel'), 'Cardiovascular') \
    .when(F.col('SMTS').isin('Muscle', 'Adipose Tissue', 'Skin'), 'Musculoesqueletico') \
    .otherwise('Visceral_Metabolico')

sex_label_col = F.when(F.col('SEX') == '1', 'Masculino').otherwise('Femenino')

meta_df = meta_df \
    .withColumn('TISSUE_GROUP', tissue_group_col) \
    .withColumn('SEX_LABEL', sex_label_col)

# normalizo el nombre de columna (en el archivo TPM los SAMPID tienen guiones reemplazados)
meta_df = meta_df.withColumn(
    'COL_NAME',
    F.regexp_replace(F.regexp_replace(F.col('SAMPID'), '-', '_'), '\\.', '_')
)

print(f'Muestras con metadatos completos: {meta_df.count():,}')
meta_df.select('SAMPID', 'SMTS', 'TISSUE_GROUP', 'SEX_LABEL').show(3, truncate=False)

Muestras con metadatos completos: 19,788
+-----------------------------+-----+--------------+---------+
|SAMPID                       |SMTS |TISSUE_GROUP  |SEX_LABEL|
+-----------------------------+-----+--------------+---------+
|GTEX-1117F-0005-SM-HL9SH     |Blood|Hematopoyetico|Femenino |
|GTEX-1117F-0011-R10b-SM-GI4VE|Brain|Nervioso      |Femenino |
|GTEX-1117F-0011-R11b-SM-GIN8R|Brain|Nervioso      |Femenino |
+-----------------------------+-----+--------------+---------+
only showing top 3 rows


In [52]:
# verifico cuántas muestras hay por grupo de tejido (esto es lo que tendría que ser balanceado en M')
meta_df.groupBy('TISSUE_GROUP').count().orderBy('TISSUE_GROUP').show()

+-------------------+-----+
|       TISSUE_GROUP|count|
+-------------------+-----+
|     Cardiovascular| 2344|
|     Hematopoyetico| 1407|
| Musculoesqueletico| 4176|
|           Nervioso| 3904|
|Visceral_Metabolico| 7957|
+-------------------+-----+



### 2.3 Carga del TPM y construcción de M

**Definición de M para esta actividad:** En el proyecto de equipo M se construyó tomando 1 de cada 100 columnas (`SAMPLE_STEP=100`) para tener tiempos de procesamiento razonables en la fase exploratoria (terminamos con ~197 muestras). Para esta actividad **expando M a 1 de cada 50 columnas** (`SAMPLE_STEP=50`, ~390 muestras) porque necesito más datos para que los modelos de ML tengan suficiente para aprender. Sigue siendo un subconjunto manejable individualmente.

In [53]:
# fijo el step de muestreo para esta actividad (más denso que el del equipo)
ACT3_SAMPLE_STEP = 50

# leo solo el header del archivo TPM para obtener los nombres de columnas (rápido)
peek = pd.read_csv(FILE_PATH, sep='\t', skiprows=2, nrows=0)
all_col_names = peek.columns.tolist()

# selecciono 1 de cada N columnas de muestra (las dos primeras son Name y Description)
selected_indices = [0, 1] + list(range(2, len(all_col_names), ACT3_SAMPLE_STEP))
selected_cols_raw = [all_col_names[i] for i in selected_indices]
clean_names = [c.replace('-', '_').replace('.', '_') for c in selected_cols_raw]
sample_cols_M = clean_names[2:]

print(f'Total de columnas en el archivo: {len(all_col_names) - 2:,}')
print(f'Columnas seleccionadas para M:   {len(sample_cols_M):,} (1/{ACT3_SAMPLE_STEP})')

Total de columnas en el archivo: 19,616
Columnas seleccionadas para M:   393 (1/50)


In [54]:
# leo el archivo TPM como texto y filtro a filas de genes (las que empiezan con ENSG)
# uso spark.read.text + Spark SQL puro para evitar lambdas en Python (problemas en Windows)
raw_df = spark.read.text(FILE_PATH)
gene_df = raw_df.filter(F.col('value').startswith('ENSG'))

# parseo cada línea por tab y elijo solo las columnas que necesito
split_col = spark_split(F.col('value'), '\t')
df_tpm_M = gene_df.select(
    *[split_col.getItem(i).alias(clean_names[idx]) for idx, i in enumerate(selected_indices)]
)

# casteo las columnas de muestra a float
for c in sample_cols_M:
    df_tpm_M = df_tpm_M.withColumn(c, F.col(c).cast(FloatType()))

print(f'M cargada: {df_tpm_M.count():,} genes x {len(sample_cols_M)} muestras')
df_tpm_M.select(clean_names[:4]).show(3)

M cargada: 59,033 genes x 393 muestras
+-----------------+-----------+------------------------+----------------------------+
|             Name|Description|GTEX_1117F_0005_SM_HL9SH|GTEX_111FC_0011_R5a_SM_GIN8L|
+-----------------+-----------+------------------------+----------------------------+
|ENSG00000223972.5|    DDX11L1|                     0.0|                         0.0|
|ENSG00000227232.5|     WASH7P|                 1.33343|                     3.44305|
|ENSG00000278267.1|  MIR6859-1|                     0.0|                    0.365439|
+-----------------+-----------+------------------------+----------------------------+
only showing top 3 rows


### 2.4 Construcción de M' por muestreo estratificado

**Por qué estratificado:** Si hiciera un muestreo aleatorio simple, podría perder o subrepresentar los grupos de tejido menos frecuentes (como Hematopoyético-Femenino, que tiene pocas muestras). Estratifico por `TISSUE_GROUP` para asegurar que las 5 clases estén presentes en M' en proporciones similares a M.

**Tamaño:** Tomo el 80% de cada grupo. Esto sigue manteniendo M' como subconjunto propio de M y deja suficientes muestras por clase para entrenar.

In [55]:
# precomputo un dict de col_name -> tissue_group para lookups rápidos
meta_lookup = {
    row['COL_NAME']: row['TISSUE_GROUP']
    for row in meta_df.select('COL_NAME', 'TISSUE_GROUP').collect()
}

# filtro las columnas de M a aquellas para las que tengo metadatos completos
sample_cols_M = [c for c in sample_cols_M if c in meta_lookup]
print(f'Columnas de M con metadatos completos: {len(sample_cols_M)}')

Columnas de M con metadatos completos: 393


In [56]:
def stratified_sample_columns(cols, lookup, fraction, seed):
    """Devuelve una sub-lista de cols estratificada por grupo de tejido."""
    rng = random.Random(seed)
    # agrupo columnas por grupo de tejido
    by_group = {}
    for c in cols:
        tg = lookup.get(c)
        if tg is None:
            continue
        by_group.setdefault(tg, []).append(c)
    # muestreo dentro de cada grupo
    sampled = []
    for tg, group_cols in by_group.items():
        n = max(1, int(round(len(group_cols) * fraction)))
        sampled.extend(rng.sample(group_cols, min(n, len(group_cols))))
    return sorted(sampled)

In [57]:
# construyo M' como el 80% de M estratificado por grupo de tejido
MPRIMA_FRACTION = 0.80
sample_cols_Mp = stratified_sample_columns(
    sample_cols_M, meta_lookup, fraction=MPRIMA_FRACTION, seed=RANDOM_SEED
)

print(f'|M|  = {len(sample_cols_M)} muestras')
print(f"|M'| = {len(sample_cols_Mp)} muestras (fraction={MPRIMA_FRACTION})")

|M|  = 393 muestras
|M'| = 315 muestras (fraction=0.8)


In [58]:
# verifico que M' mantiene la distribución de clases de M
def count_by_group(cols, lookup):
    counts = {}
    for c in cols:
        tg = lookup.get(c)
        counts[tg] = counts.get(tg, 0) + 1
    return counts

counts_M  = count_by_group(sample_cols_M,  meta_lookup)
counts_Mp = count_by_group(sample_cols_Mp, meta_lookup)

pd.DataFrame({'M': counts_M, "M'": counts_Mp}).fillna(0).astype(int).sort_index()

,M,M'
Cardiovascular,47,38
Hematopoyetico,19,15
Musculoesqueletico,72,58
Nervioso,73,58
Visceral_Metabolico,182,146


In [59]:
# filtro df_tpm para quedarme solo con las columnas de M'
df_tpm_Mp = df_tpm_M.select(['Name', 'Description'] + sample_cols_Mp)
print(f"df_tpm_Mp: {df_tpm_Mp.count():,} genes x {len(sample_cols_Mp)} muestras")

df_tpm_Mp: 59,033 genes x 315 muestras


### 2.5 Selección de los top 500 genes por varianza

**Por qué varianza:** Genes con varianza ~0 no aportan información discriminativa (su valor es prácticamente constante entre tejidos). Quedarme con los 500 de mayor varianza es una forma sencilla de reducir dimensionalidad y enfocarme en los genes que probablemente distingan tejidos.

**Por qué 500:** Es un número común en análisis de expresión génica para feature selection no supervisada. Equilibra suficiente información biológica con tractabilidad para el modelo dado el tamaño de M'.

In [60]:
def add_variance_column(df, sample_cols):
    """Agrega columna 'variance' = varianza poblacional de cada gen sobre sample_cols.

    Uso F.aggregate sobre un F.array en vez de sum(F.col(c) for c in cols).
    Razón: con muchas columnas (300+), el sum() genera un árbol de + anidados de
    300 niveles de profundidad y Catalyst corta la resolución en 100 iteraciones
    (error 'Max iterations reached for batch Resolution'). F.aggregate produce un
    plan plano (un solo nodo) y siempre resuelve.

    Para la varianza uso la fórmula computacional Var(X) = E[X^2] - E[X]^2,
    así con un solo recorrido del array obtengo sum_x y sum_x2 sin necesidad
    de una segunda pasada que referencie la media.
    """
    n = len(sample_cols)
    # caso a double para asegurar tipos consistentes con el acumulador
    arr = F.array(*[F.col(c).cast('double') for c in sample_cols])
    sum_x  = F.aggregate(arr, F.lit(0.0), lambda acc, x: acc + x)
    sum_x2 = F.aggregate(arr, F.lit(0.0), lambda acc, x: acc + x * x)
    mean = sum_x / n
    variance = (sum_x2 / n) - (mean * mean)
    return df.withColumn('variance', variance)

In [61]:
# calculo la varianza por gen y selecciono los top 500
df_with_var = add_variance_column(df_tpm_Mp, sample_cols_Mp)

TOP_K_GENES = 500
top_genes_df = df_with_var.orderBy(F.col('variance').desc()).limit(TOP_K_GENES)
top_gene_names = [row['Name'] for row in top_genes_df.select('Name').collect()]

print(f'Top {TOP_K_GENES} genes seleccionados por varianza.')
top_genes_df.select('Name', 'Description', 'variance').show(10, truncate=False)

Top 500 genes seleccionados por varianza.


+------------------+-----------+--------------------+
|Name              |Description|variance            |
+------------------+-----------+--------------------+
|ENSG00000244734.4 |HBB        |2.373455494264391E9 |
|ENSG00000210082.2 |MT-RNR2    |7.122656336882546E8 |
|ENSG00000198804.2 |MT-CO1     |5.708790924918897E8 |
|ENSG00000259384.7 |GH1        |5.126145077597021E8 |
|ENSG00000198712.1 |MT-CO2     |4.402625888789747E8 |
|ENSG00000198938.2 |MT-CO3     |4.0252609609759974E8|
|ENSG00000198886.2 |MT-ND4     |3.5979200256172156E8|
|ENSG00000198899.2 |MT-ATP6    |3.289978795336349E8 |
|ENSG00000188536.13|HBA2       |3.2350797701152873E8|
|ENSG00000275896.7 |PRSS2      |2.6315697313901508E8|
+------------------+-----------+--------------------+
only showing top 10 rows


In [62]:
# filtro df_tpm_Mp para quedarme solo con los genes top
df_tpm_top = df_tpm_Mp.filter(F.col('Name').isin(top_gene_names))
print(f'Forma del subset top-genes: {df_tpm_top.count():,} genes x {len(sample_cols_Mp)} muestras')

Forma del subset top-genes: 500 genes x 315 muestras


### 2.6 Transposición a formato ML

El archivo TPM tiene genes como filas y muestras como columnas, pero para entrenar un modelo necesito el formato inverso: **una fila por muestra** y **una columna por feature (gen)**. Además agrego la columna `TISSUE_GROUP` como etiqueta.

Como el subset top-genes ya es chico (500 genes x ~310 muestras ~ 155K celdas), traigo el DataFrame a pandas, transpongo, y lo regreso a Spark. Mucho más simple que pivot/unpivot en Spark puro.

In [63]:
# colecto el subset a pandas para transponerlo
pdf_top = df_tpm_top.toPandas()
print(f'pdf_top shape (antes de transponer): {pdf_top.shape}')
pdf_top.head(3)

pdf_top shape (antes de transponer): (500, 317)


,Name,Description,GTEX_1117F_0005_SM_HL9SH,GTEX_111FC_0011_R5a_SM_GIN8L,GTEX_111YS_1426_SM_5GID8,GTEX_1128S_2626_SM_5H11Z,GTEX_1192W_0226_SM_5EGGT,GTEX_11DXW_1226_SM_5H133,GTEX_11DYG_0826_SM_5N9GH,GTEX_11EMC_0826_SM_59862,...,GTEX_ZT9W_2226_SM_57WFU,GTEX_ZTSS_0326_SM_5987M,GTEX_ZTX8_1726_SM_51MSB,GTEX_ZV68_0926_SM_59HK7,GTEX_ZVT3_0008_SM_51MRI,GTEX_ZVZO_0126_SM_5A5L9,GTEX_ZWKS_0326_SM_5NQ7G,GTEX_ZY6K_1226_SM_5GZYL,GTEX_ZYFG_0326_SM_5E45Z,GTEX_ZYY3_2526_SM_GMXAZ
0,ENSG00000225972.1,MTND1P23,0.52891,5.945260,21.97480,5.913790,33.064899,7.455580,447.842987,25.064600,...,19.406700,16.331499,1893.760010,22.639601,16.700899,3495.679932,9.895880,10.124700,10.070600,4.565730
1,ENSG00000225630.1,MTND2P28,9.17182,1197.099976,1114.47998,525.908020,1048.300049,729.210999,432.334991,945.104004,...,552.679993,506.156006,631.255981,4135.370117,213.845001,608.864014,1158.660034,773.956970,451.075012,735.953003
2,ENSG00000237973.1,MTCO1P12,1.63962,64.429604,144.83400,28.495501,121.941002,19.768200,213.524002,73.005096,...,58.433399,112.762001,140.509995,161.932007,52.319000,2513.830078,49.237301,429.191986,39.293098,39.515701


In [64]:
# transpongo: descarto Description, uso Name como índice, transpongo
pdf_T = pdf_top.drop(columns=['Description']).set_index('Name').T

# renombro columnas a gene_0001, gene_0002, ... para evitar problemas con puntos en nombres de columnas en Spark
gene_name_map = {orig: f'gene_{i:04d}' for i, orig in enumerate(pdf_T.columns)}
pdf_T.columns = [gene_name_map[c] for c in pdf_T.columns]

# el índice ahora son los SAMPLE IDs (col names limpios). Lo paso a columna.
pdf_T.index.name = 'COL_NAME'
pdf_T = pdf_T.reset_index()

print(f'pdf_T shape (después de transponer): {pdf_T.shape}')
pdf_T.head(3)

pdf_T shape (después de transponer): (315, 501)


,COL_NAME,gene_0000,gene_0001,gene_0002,gene_0003,gene_0004,gene_0005,gene_0006,gene_0007,gene_0008,...,gene_0490,gene_0491,gene_0492,gene_0493,gene_0494,gene_0495,gene_0496,gene_0497,gene_0498,gene_0499
0,GTEX_1117F_0005_SM_HL9SH,0.52891,9.171820,1.639620,126.876999,137.600006,0.230122,0.527239,0.698854,0.556642,...,1063.020020,536.492004,1054.939941,1208.040039,259.591003,290.156006,842.604004,102.485001,95.753799,430.554993
1,GTEX_111FC_0011_R5a_SM_GIN8L,5.94526,1197.099976,64.429604,5122.870117,705.091980,0.261577,0.035673,0.203687,1.190470,...,75409.203125,20480.000000,51267.601562,43454.699219,19408.099609,24845.000000,47074.101562,9463.129883,8178.919922,35858.601562
2,GTEX_111YS_1426_SM_5GID8,21.97480,1114.479980,144.834000,7035.509766,377.851990,6.373970,0.294646,0.696157,3.323880,...,84388.203125,28478.300781,59160.898438,58988.898438,21124.400391,17116.699219,51715.199219,9170.740234,13717.500000,36179.300781


In [65]:
# convierto la matriz transpuesta de vuelta a Spark
df_features = spark.createDataFrame(pdf_T)

# uno con metadatos para traer la etiqueta TISSUE_GROUP
df_ml = df_features.join(
    meta_df.select('COL_NAME', 'TISSUE_GROUP'),
    on='COL_NAME',
    how='inner'
)

print(f'df_ml shape: {df_ml.count()} muestras x {len(df_ml.columns)} columnas (incluye COL_NAME + label + 500 genes)')
df_ml.select('COL_NAME', 'TISSUE_GROUP', 'gene_0000', 'gene_0001').show(5)

df_ml shape: 315 muestras x 502 columnas (incluye COL_NAME + label + 500 genes)
+--------------------+-------------------+------------------+------------------+
|            COL_NAME|       TISSUE_GROUP|         gene_0000|         gene_0001|
+--------------------+-------------------+------------------+------------------+
|GTEX_11LCK_1226_S...| Musculoesqueletico|11.603799819946289| 808.0499877929688|
|GTEX_11ZUS_0011_R...|           Nervioso|10.611800193786621|1139.3199462890625|
|GTEX_1117F_0005_S...|     Hematopoyetico|0.5289099812507629| 9.171819686889648|
|GTEX_11P82_0826_S...|Visceral_Metabolico| 5.701250076293945| 621.4739990234375|
|GTEX_11TT1_1526_S...|     Cardiovascular| 6.917409896850586| 352.3399963378906|
+--------------------+-------------------+------------------+------------------+
only showing top 5 rows


## 3. Preparación del conjunto de entrenamiento y prueba

### 3.1 Justificación de la proporción 80/20

**Proporción elegida: 80% entrenamiento / 20% prueba.**

Justificación:
- Mi M' tiene alrededor de 310 muestras. Con 5 clases (grupos de tejido), un 80/20 da ~248 train / ~62 test, lo que deja ~12 muestras de test por clase en promedio - suficiente para que las métricas no sean ruidosas, sin sacrificar demasiado del set de entrenamiento.
- Con 500 features y dataset relativamente pequeño, necesito maximizar los datos disponibles para entrenar y evitar que el modelo no tenga ejemplos suficientes por clase. 70/30 sería más conservador para test pero dejaría menos para entrenar.
- Uso `seed=42` para reproducibilidad.

**Técnica para evitar sesgos:**
Uso `randomSplit([0.8, 0.2], seed=RANDOM_SEED)` de PySpark. Después de splittear, verifico que la distribución de clases en train y test sea similar (que no haya quedado un grupo de tejido completamente en uno de los lados). Si la distribución quedara muy desbalanceada habría que considerar split estratificado manual; con 5 clases relativamente bien distribuidas en M', el random split debería ser suficiente.

In [66]:
# convierto la etiqueta de texto a índice numérico (requerido por los clasificadores de Spark ML)
indexer = StringIndexer(inputCol='TISSUE_GROUP', outputCol='label')
indexer_model = indexer.fit(df_ml)
df_indexed = indexer_model.transform(df_ml)

# guardo el mapeo índice -> nombre para usarlo en la interpretación
label_names = indexer_model.labels  # list[str], idx -> name
print('Mapeo de etiquetas (índice -> grupo):')
for i, name in enumerate(label_names):
    print(f'  {i}: {name}')

Mapeo de etiquetas (índice -> grupo):
  0: Visceral_Metabolico
  1: Musculoesqueletico
  2: Nervioso
  3: Cardiovascular
  4: Hematopoyetico


In [67]:
# ensamblo las 500 columnas de genes en un solo vector de features
gene_feature_cols = [c for c in df_indexed.columns if c.startswith('gene_')]
print(f'Número de features: {len(gene_feature_cols)}')

assembler = VectorAssembler(inputCols=gene_feature_cols, outputCol='features_raw')
df_assembled = assembler.transform(df_indexed)
df_assembled.select('COL_NAME', 'label', 'features_raw').show(3, truncate=80)

Número de features: 500


+----------------------------+-----+--------------------------------------------------------------------------------+
|                    COL_NAME|label|                                                                    features_raw|
+----------------------------+-----+--------------------------------------------------------------------------------+
|    GTEX_11LCK_1226_SM_5Q5AM|  1.0|[11.603799819946289,808.0499877929688,133.36300659179688,5064.10009765625,72....|
|GTEX_11ZUS_0011_R3b_SM_GIN95|  2.0|[10.611800193786621,1139.3199462890625,74.37480163574219,4522.919921875,396.9...|
|    GTEX_1117F_0005_SM_HL9SH|  4.0|[0.5289099812507629,9.171819686889648,1.6396199464797974,126.87699890136719,1...|
+----------------------------+-----+--------------------------------------------------------------------------------+
only showing top 3 rows


In [68]:
# split 80/20 con seed fija para reproducibilidad
train_df, test_df = df_assembled.randomSplit([0.8, 0.2], seed=RANDOM_SEED)
train_df.cache()
test_df.cache()
print(f'Train: {train_df.count()} muestras')
print(f'Test:  {test_df.count()} muestras')

Train: 257 muestras


Test:  58 muestras


In [69]:
# verifico que la distribución de clases en train/test sea similar
def class_distribution(df, name):
    counts = df.groupBy('TISSUE_GROUP').count().toPandas().set_index('TISSUE_GROUP')
    counts.columns = [name]
    return counts

dist = pd.concat([class_distribution(train_df, 'train'),
                  class_distribution(test_df, 'test')], axis=1).fillna(0).astype(int)
dist['train_pct'] = (dist['train'] / dist['train'].sum() * 100).round(1)
dist['test_pct']  = (dist['test']  / dist['test'].sum()  * 100).round(1)
dist.sort_index()

,train,test,train_pct,test_pct
TISSUE_GROUP,,,,
Cardiovascular,32,6,12.5,10.3
Hematopoyetico,11,4,4.3,6.9
Musculoesqueletico,48,10,18.7,17.2
Nervioso,46,12,17.9,20.7
Visceral_Metabolico,120,26,46.7,44.8


La distribución se mantuvo bien parecida entre train y test. Visceral_Metabolico domina con ~46% en los dos, y las demás clases no se desviaron más de ~3 puntos porcentuales. El `randomSplit` funcionó bien aquí, no tuve que splittear estratificado a mano.

El que sí me preocupa es Hematopoyético: quedó con solo 4 muestras en test. Eso es tan poco que una sola predicción mal hecha le tira el F1 de esa clase en 25 puntos. Si quisiera métricas confiables para clases chicas haría k-fold cross-validation o split estratificado manual con `sampleBy`. Para esta actividad lo acepto y lo menciono después en las conclusiones.

## 4. Construcción de modelos de aprendizaje supervisado

### 4.1 Función de entrenamiento y evaluación

Como voy a entrenar dos modelos y aplicar las mismas métricas a ambos, encapsulo el train+eval en una función para reutilizarla.

In [70]:
# defino una vez los evaluadores multi-clase
eval_accuracy = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='accuracy'
)
eval_f1 = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='f1'
)
eval_precision = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='weightedPrecision'
)
eval_recall = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='weightedRecall'
)

def train_and_evaluate(pipeline, train, test, model_name):
    """Entrena el pipeline en train, evalúa en test y devuelve métricas + objetos."""
    print(f'>>> Entrenando {model_name}...')
    model = pipeline.fit(train)
    preds = model.transform(test)
    metrics = {
        'model': model_name,
        'accuracy': eval_accuracy.evaluate(preds),
        'f1':       eval_f1.evaluate(preds),
        'precision': eval_precision.evaluate(preds),
        'recall':   eval_recall.evaluate(preds),
    }
    print(f'    accuracy:  {metrics["accuracy"]:.4f}')
    print(f'    f1 (weighted): {metrics["f1"]:.4f}')
    print(f'    precision (weighted): {metrics["precision"]:.4f}')
    print(f'    recall (weighted):    {metrics["recall"]:.4f}')
    return {'name': model_name, 'model': model, 'predictions': preds, 'metrics': metrics}

### 4.2 Modelo 1: Random Forest Classifier

**Supuestos del modelo:**
- Los árboles del ensemble no requieren escalado de features (las divisiones son sobre umbrales por feature). Por eso para RF uso directamente `features_raw`.
- `numTrees=100` da estabilidad sin tardar demasiado.
- `maxDepth=10` deja a los árboles crecer lo suficiente para captar interacciones gen-gen, pero limita overfitting.
- `seed=RANDOM_SEED` para reproducibilidad (RF tiene aleatoriedad en bootstrap y selección de features).

**Métrica principal:** F1 weighted, porque mide precision/recall por clase y pondera por soporte. Es más robusta que accuracy si hay clases desbalanceadas.

In [71]:
# construyo el clasificador RF
rf = RandomForestClassifier(
    featuresCol='features_raw',
    labelCol='label',
    numTrees=100,
    maxDepth=10,
    seed=RANDOM_SEED,
)

# el assembler ya está aplicado en df_assembled, así que el pipeline solo tiene el modelo
rf_pipeline = Pipeline(stages=[rf])

rf_result = train_and_evaluate(rf_pipeline, train_df, test_df, model_name='RandomForest')

>>> Entrenando RandomForest...


26/05/24 22:07:05 WARN DAGScheduler: Broadcasting large task binary with size 1007.0 KiB


    accuracy:  0.9828
    f1 (weighted): 0.9825
    precision (weighted): 0.9834
    recall (weighted):    0.9828


In [72]:
# extraigo importancias de features (qué genes pesan más en las predicciones)
rf_model = rf_result['model'].stages[-1]  # último stage del pipeline es el RF entrenado
importances = rf_model.featureImportances.toArray()

# armo un dataframe gen -> importancia y muestro los top 15
inverse_gene_map = {v: k for k, v in gene_name_map.items()}
fi_df = pd.DataFrame({
    'gene_col': gene_feature_cols,
    'gene_id':  [inverse_gene_map[c] for c in gene_feature_cols],
    'importance': importances,
}).sort_values('importance', ascending=False)

print('Top 15 genes más importantes para Random Forest:')
fi_df.head(15)

Top 15 genes más importantes para Random Forest:


,gene_col,gene_id,importance
49,gene_0049,ENSG00000160781.17,0.026459
167,gene_0167,ENSG00000180354.16,0.022272
396,gene_0396,ENSG00000197971.16,0.019254
326,gene_0326,ENSG00000159251.8,0.016979
350,gene_0350,ENSG00000087250.9,0.016340
161,gene_0161,ENSG00000272398.6,0.013917
482,gene_0482,ENSG00000182492.16,0.012594
283,gene_0283,ENSG00000111341.10,0.012559
44,gene_0044,ENSG00000196154.12,0.011291
450,gene_0450,ENSG00000198959.12,0.011165


RF salió mejor de lo que esperaba: accuracy 0.9828, F1 0.9825. Solo 1 error en 58 muestras de test.

Lo que me llamó la atención es que la importancia de features está muy repartida. El gen más importante pesa apenas 2.6% y los siguientes 14 entre 1-2% cada uno. RF no se está apoyando en un solo marcador, está combinando muchos genes para decidir. Eso lo hace más robusto a ruido en cualquier gen individual.

Entre los top aparece `ENSG00000197971` que es **MBP** (myelin basic protein), marcador clásico del sistema nervioso. Que un gen así aparezca alto me da confianza de que el modelo se está apoyando en biología real y no en correlaciones espurias.

### 4.3 Modelo 2: Regresión Logística Multinomial

**Por qué este como comparación:** Es lineal, paradigma completamente distinto a RF. Si LR alcanza accuracy parecida a RF, significa que las relaciones gen-tejido son aproximadamente lineales en escala TPM y no se requiere la flexibilidad no lineal de los árboles. Si RF gana por mucho, las interacciones no lineales y umbrales por gen importan.

**Supuestos:**
- Modelo lineal en los features escalados.
- A diferencia de RF, **requiere escalado** porque cuando combinas features con magnitudes muy distintas (la expresión TPM va de 0 a decenas de miles), los coeficientes y la regularización quedan dominados por los genes de mayor magnitud. Aplico `StandardScaler` (media 0, std 1).
- `family='multinomial'` para clasificación multi-clase (en vez de OneVsRest).
- `maxIter=50` y `regParam=0.0` (sin regularización adicional). Si hubiera overfitting subiría `regParam`.

In [73]:
# escalado de features (necesario para LR)
scaler = StandardScaler(
    inputCol='features_raw',
    outputCol='features_scaled',
    withMean=True,
    withStd=True,
)

lr = LogisticRegression(
    featuresCol='features_scaled',
    labelCol='label',
    family='multinomial',
    maxIter=50,
    regParam=0.0,
)

lr_pipeline = Pipeline(stages=[scaler, lr])

lr_result = train_and_evaluate(lr_pipeline, train_df, test_df, model_name='LogisticRegression')

>>> Entrenando LogisticRegression...


    accuracy:  0.9483
    f1 (weighted): 0.9492
    precision (weighted): 0.9550
    recall (weighted):    0.9483


LR también anduvo muy bien: accuracy 0.9483, F1 0.9492. Cometió 3 errores en lugar de 1.

No hubo warnings de convergencia con `maxIter=50`, así que se entrenó completo sin necesitar más iteraciones. Como llegó a >94% sin capacidad no lineal, me dice que las relaciones gen-tejido son en su mayoría linealmente separables después de escalar los TPM. RF gana por ~3 puntos, lo que sugiere que sí hay algunas interacciones gen-gen que captura mejor, pero la mayor parte de la señal discriminativa ya es lineal.

### 4.4 Comparación lado a lado

In [74]:
# armo tabla resumen comparando los dos modelos
comparison = pd.DataFrame([rf_result['metrics'], lr_result['metrics']])
comparison = comparison.set_index('model')
comparison = comparison.round(4)
comparison

,accuracy,f1,precision,recall
model,,,,
RandomForest,0.9828,0.9825,0.9834,0.9828
LogisticRegression,0.9483,0.9492,0.9550,0.9483


In [75]:
def confusion_matrix(predictions, label_names):
    """Devuelve la matriz de confusión como DataFrame (filas=real, cols=predicho)."""
    cm = predictions.groupBy('label', 'prediction').count().toPandas()
    cm_pivot = cm.pivot(index='label', columns='prediction', values='count').fillna(0).astype(int)
    cm_pivot.index = [label_names[int(i)] for i in cm_pivot.index]
    cm_pivot.columns = [label_names[int(i)] for i in cm_pivot.columns]
    cm_pivot.index.name = 'real'
    cm_pivot.columns.name = 'predicho'
    return cm_pivot

In [76]:
print('Matriz de confusión - Random Forest:')
confusion_matrix(rf_result['predictions'], label_names)

Matriz de confusión - Random Forest:


predicho,Visceral_Metabolico,Musculoesqueletico,Nervioso,Cardiovascular,Hematopoyetico
real,,,,,
Visceral_Metabolico,26,0,0,0,0
Musculoesqueletico,1,9,0,0,0
Nervioso,0,0,12,0,0
Cardiovascular,0,0,0,6,0
Hematopoyetico,0,0,0,0,4


In [77]:
print('Matriz de confusión - Logistic Regression:')
confusion_matrix(lr_result['predictions'], label_names)

Matriz de confusión - Logistic Regression:


predicho,Visceral_Metabolico,Musculoesqueletico,Nervioso,Cardiovascular,Hematopoyetico
real,,,,,
Visceral_Metabolico,25,1,0,0,0
Musculoesqueletico,0,8,0,2,0
Nervioso,0,0,12,0,0
Cardiovascular,0,0,0,6,0
Hematopoyetico,0,0,0,0,4


**Quién ganó:** RF, por un margen chico pero consistente en las 4 métricas. 1 error contra 3 en test.

**El error que cometió RF:** 1 muestra de Musculoesquelético predicha como Visceral_Metabolico. Todas las demás clases las clavó (Visceral 26/26, Nervioso 12/12, Cardio 6/6, Hemato 4/4).

**Los errores de LR:**
- 1 Visceral predicho como Musculo (el mismo eje de confusión que RF pero al revés).
- 2 muestras de Musculo predichas como Cardiovascular.

**Por qué tiene sentido biológicamente:** Las dos confusiones se explican bien con cómo el equipo definió los grupos.

1. *Musculo <-> Visceral_Metabolico:* En el mapeo, "Musculoesquelético" incluye Adipose Tissue y Skin además de Muscle. La piel y el tejido adiposo son metabólicamente activos y comparten patrones de expresión con tejidos viscerales como hígado o intestino. No es ruido, es ambigüedad real en la definición de los grupos.
2. *Musculo -> Cardiovascular en LR:* Esta es la confusión más biológicamente lógica de todas. El músculo esquelético y el músculo cardíaco (corazón) comparten genes de miocitos (actinas, miosinas, troponinas). LR, que es lineal, no logró separarlos tan bien como RF. Esta diferencia es probablemente la razón principal por la que RF ganó.

**Tejidos fáciles:** Nervioso, Hematopoyético y Cardiovascular salieron 100% bien en ambos modelos. Tiene sentido: Nervioso tiene marcadores muy específicos (MBP entre otros), Hematopoyético está dominado por HBB y HBA2 (los dos en el top 10 de varianza), y Cardiovascular tiene la firma del corazón bastante distinta de todo lo demás (excepto del músculo esquelético, ahí está la confusión que vimos).

**Caveat importante:** El test set tiene solo 58 muestras. Una sola predicción equivocada mueve la accuracy en 1.7 puntos. Para afirmar que RF > LR con significancia estadística haría k-fold cross-validation con k=5 o k=10. Sin eso, el ranking es plausible pero ruidoso.

## 5. Conclusiones

**Lo que hice:** Implementé un pipeline completo de aprendizaje supervisado en PySpark sobre el dataset GTEx. Construí M (393 muestras, 1 de cada 50 columnas del dataset completo) y M' (315 muestras por muestreo estratificado al 80% por grupo de tejido). Seleccioné los top 500 genes por varianza usando `F.aggregate` sobre arrays (importante: la versión naive con `sum(F.col)` se rompe con 300+ columnas porque Catalyst no resuelve árboles tan profundos). Splittéé 80/20 y entrené dos modelos en paralelo: Random Forest y Logistic Regression multinomial.

**Lo que aprendí:** Los dos modelos clasifican grupo de tejido con accuracy >94%, lo que confirma la hipótesis biológica de que el perfil de expresión génica es altamente discriminativo entre tejidos. RF llegó a 0.98 y LR a 0.95. La diferencia me dice que sí hay algunas interacciones gen-gen no lineales que RF captura mejor (concretamente, la separación entre músculo esquelético y músculo cardíaco), pero la mayor parte de la información discriminativa ya es linealmente separable después del escalado. Los top genes seleccionados por varianza (HBB, hemoglobinas, mitocondriales, GH1) son marcadores tejido-específicos muy conocidos en la literatura, así que el método de feature selection se validó solo - no estoy entrenando con ruido.

**Lo que no hice y podría ser trabajo/test futuro:** Cross-validation para estabilizar las métricas (especialmente para Hematopoyético con n=4 en test, donde una sola predicción mal hecha cambia el F1 de esa clase en 25 puntos). Tuning de hiperparámetros - RF con `maxDepth=10` y LR sin regularización funcionaron muy bien con estos defaults, pero no descarto que regularizar más generalice mejor en datasets más grandes. Feature selection multivariada o por mutual information en vez de varianza univariada (la varianza ignora correlaciones entre genes).

**Para qué sirve esto:** El pipeline está listo para el caso de uso del proyecto de equipo :) Si entreno el modelo en GTEx (línea base sana terrestre) y le paso muestras de astronautas u otras condiciones extremas, las muestras donde el modelo asigne probabilidad baja a la clase de tejido esperada serían señal de que el perfil de expresión se desvió de lo normal. Esa es la conexión directa entre esta actividad y la pregunta original que estamos investigando.

## Referencias

1. GTEx Consortium. (2020). The GTEx Consortium atlas of genetic regulatory effects across human tissues. *Science*. https://doi.org/10.1126/science.aaz1776
2. Apache Spark. (2025). MLlib Guide - Classification and regression. https://spark.apache.org/docs/latest/ml-classification-regression.html
3. Jiménez, E. C. (2025). Aprendizaje supervisado: ventajas, limitaciones y su papel en la próxima generación de tecnologías. MasScience.

## Declaración de uso de Inteligencia Artificial

Anthropic. (2026). *Claude Opus 4.7* [Modelo de lenguaje grande], utilizado para limpieza de markdown, explicación de conceptos y revisión final. https://claude.ai

*La responsabilidad final sobre el contenido entregado recae en la autora. La selección de variables, decisión sobre el algoritmo, justificación de proporciones, e interpretación de los resultados son propias.*